# 03 — Topics des prises de parole (LDA)

Objectif : regrouper les prises de parole (*chunks* de `01_chunks`) qui parlent de sujets proches, et donner un nom à chaque groupe.

Méthode :
- **corpus** : les prises de parole d'au moins `MIN_WORDS` mots, sans le nom de l'orateur qui les ouvre ; les répliques courtes (« Très bien ! », « La parole est à M. X. ») ne disent rien du sujet ;
- **sac de mots** : mots d'au moins 3 lettres, hors mots-outils et vocabulaire de procédure (`debats.topics.STOPWORDS_FR`), présents dans au moins 5 prises de parole et au plus 40 % d'entre elles ; les pluriels réguliers sont fusionnés avec leur singulier (« blés » → « blé ») ;
- **LDA** (`sklearn`) à `K` topics ; chaque prise de parole est rattachée à son topic dominant ;
- **nom d'un topic** : ses 3 mots les plus caractéristiques (*relevance* de Sievert & Shirley, λ = 0,6), précédés de la rubrique d'ordre du jour sur-représentée parmi ses prises de parole, ou de « Procédure » si ses mots-clés sont des mots de procédure (dépôt, renvoi, consultation…) ;
- **contrôle** : les prises de parole qui ont un plongement (`data/chunk_embeddings.jsonl.gz`) doivent être plus proches entre elles au sein d'un même topic qu'entre topics.

Sortie : `output/topics/topics.csv` (un topic par ligne) et `output/topics/chunk_topics.csv` (topic de chaque prise de parole).

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.decomposition import LatentDirichletAllocation

from debats import OUTPUT_DIR
from debats.chunks import load_chunks, load_embeddings, strip_speaker
from debats.topics import fold_plurals, make_vectorizer, name_topics, short_sections, top_words, umass_coherence

pd.set_option("display.max_colwidth", 120)

MIN_WORDS = 40  # longueur minimale d'une prise de parole (sans le nom de l'orateur)
K = 20  # nombre de topics
SEED = 0

## Corpus

In [ ]:
chunks = load_chunks()
chunks["body"] = chunks.text.map(strip_speaker)
chunks["n_words"] = chunks.body.str.split().str.len()
chunks["rubrique"] = short_sections(chunks.section)

corpus = chunks[chunks.n_words >= MIN_WORDS].reset_index(drop=True)
print(f"{len(corpus)} prises de parole d'au moins {MIN_WORDS} mots sur {len(chunks)}")

In [ ]:
corpus.groupby(["chamber", "date"]).size().unstack(0).fillna(0).astype(int).T

## Sac de mots

In [ ]:
vectorizer = make_vectorizer()
X, vocab = fold_plurals(vectorizer.fit_transform(corpus.body), vectorizer.get_feature_names_out())
word_freq = np.asarray(X.sum(axis=0)).ravel()
print(X.shape, "(prises de parole × mots)")
pd.Series(word_freq, index=vocab).nlargest(30)

## Nombre de topics

Cohérence UMass moyenne (plus proche de 0 = les mots-clés d'un topic co-occurrent davantage dans les mêmes prises de parole) et perplexité (plus bas = meilleur ajustement) pour quelques valeurs de `K`. Les deux varient peu : `K` est surtout un choix de granularité.

In [ ]:
scores = []
for k in (10, 15, 20, 25, 30):
    model = LatentDirichletAllocation(k, learning_method="batch", max_iter=30, random_state=SEED, n_jobs=-1).fit(X)
    scores.append({"K": k, "cohérence UMass": umass_coherence(model.components_, X).mean(), "perplexité": model.perplexity(X)})
pd.DataFrame(scores).set_index("K").round(3)

## Modèle

In [ ]:
lda = LatentDirichletAllocation(K, learning_method="batch", max_iter=50, random_state=SEED, n_jobs=-1).fit(X)
doc_topic = lda.transform(X)
corpus["topic"] = doc_topic.argmax(axis=1)
corpus["topic_score"] = doc_topic.max(axis=1)
corpus.topic_score.describe()

## Nommage

In [ ]:
keywords = top_words(lda.components_, vocab, word_freq)
topics = name_topics(keywords, doc_topic, corpus.rubrique)
topics["coherence"] = umass_coherence(lda.components_, X)
topics["chambre"] = corpus[corpus.chamber == "chambre"].topic.value_counts()
topics["senat"] = corpus[corpus.chamber == "senat"].topic.value_counts()
topics = topics.fillna({"chambre": 0, "senat": 0}).astype({"chambre": int, "senat": int})
topics.sort_values("n_docs", ascending=False)[
    ["name", "n_docs", "chambre", "senat", "section_share", "coherence", "keywords"]
]

Les prises de parole les plus typiques de chaque topic (score du topic dominant le plus élevé) :

In [ ]:
examples = (
    corpus.sort_values("topic_score", ascending=False)
    .groupby("topic")
    .head(2)
    .assign(name=lambda d: d.topic.map(topics.name), extrait=lambda d: d.text.str[:300])
    .sort_values(["topic", "topic_score"], ascending=[True, False])
)
examples[["topic", "name", "topic_score", "date", "chamber", "extrait"]]

## Topics au fil des séances

Nombre de prises de parole de chaque topic par date (Chambre et Sénat ensemble).

In [ ]:
counts = corpus.assign(name=corpus.topic.map(topics.name)).pivot_table(
    index="name", columns="date", values="text", aggfunc="size", fill_value=0
)
counts = counts.loc[counts.sum(axis=1).sort_values(ascending=False).index]
fig = px.imshow(
    counts,
    x=[f"{d:%d/%m}" for d in counts.columns],
    color_continuous_scale="Blues",
    aspect="auto",
    labels={"x": "date", "y": "topic", "color": "prises de parole"},
    height=650,
)
fig.update_layout(template="plotly_white", yaxis_title=None, margin={"l": 10, "r": 10, "t": 30, "b": 10})
fig.show()

## Contrôle par les plongements

Seule une partie des prises de parole a un plongement (`data/chunk_embeddings.jsonl.gz`). Pour celles du corpus, on compare la similarité cosinus moyenne entre prises de parole d'un même topic et de topics différents : un topic cohérent sémantiquement a une similarité interne supérieure à la similarité externe.

In [ ]:
keys, embeddings = load_embeddings()
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
position = pd.Series(range(len(corpus)), index=pd.Index(corpus.key, tupleize_cols=False))
in_corpus = keys.isin(position.index)
labels = corpus.topic.to_numpy()[position[keys[in_corpus]].to_numpy()]
sim = embeddings[in_corpus] @ embeddings[in_corpus].T
print(f"{in_corpus.sum()} plongements sur {len(keys)} dans le corpus")

same = labels[:, None] == labels[None, :]
print(f"similarité moyenne : même topic {sim[same & ~np.eye(len(labels), dtype=bool)].mean():.3f}, topics différents {sim[~same].mean():.3f}")

sizes = pd.Series(labels).value_counts()
rows = []
for k in sizes[sizes >= 2].index:  # au moins deux plongements pour une similarité interne
    inside = labels == k
    block = sim[np.ix_(inside, inside)]
    rows.append(
        {
            "topic": k,
            "name": topics.name[k],
            "n_plongements": sizes[k],
            "sim_interne": block[~np.eye(len(block), dtype=bool)].mean(),
            "sim_externe": sim[np.ix_(inside, ~inside)].mean(),
        }
    )
check = pd.DataFrame(rows).set_index("topic")
check["écart"] = check.sim_interne - check.sim_externe
check.sort_values("écart", ascending=False).round(3)

## Écriture

In [ ]:
out_dir = OUTPUT_DIR / "topics"
out_dir.mkdir(parents=True, exist_ok=True)
topics.assign(keywords=topics.keywords.str.join(", ")).drop(columns="top_sections").to_csv(out_dir / "topics.csv")
corpus.assign(topic_name=corpus.topic.map(topics.name))[
    ["session", "chamber", "date", "chunk", "speaker", "rubrique", "topic", "topic_name", "topic_score"]
].to_csv(out_dir / "chunk_topics.csv", index=False)
print("→", out_dir)